# DX 603: Project Milestone Two: Modeling and Feature Engineering

### Due: Sunday July 26 @ 11:59PM (with grace period of 2 hours & 1 minute)

### Overview

In Milestone 1, you explored the Zillow dataset, cleaned the data, and developed hypotheses about how preprocessing and feature engineering might improve predictive performance.

In this milestone, you will  develop, evaluate, and refine several machine learning models using those ideas. Rather than simply searching for the best algorithm, you will follow an iterative modeling workflow by:

1. Establishing baseline performance using several regression models.
2. Testing the preprocessing and feature engineering ideas proposed in Milestone 1.
3. Refining the feature set through feature selection.
4. Optimizing model performance through hyperparameter tuning.
5. Comparing the evolution of your models and selecting a final model to evaluate on the held-out test set.

Throughout this milestone, use **repeated 5-fold cross-validation (5 repeats)** to guide your modeling decisions. The held-out test set should be used only once, after all modeling decisions have been completed.




In [1]:
# ===================================
# Useful Imports: Add more as needed
# ===================================

# Standard Libraries
import os
import time
import math
import io
import zipfile
import requests
from urllib.parse import urlparse
from itertools import chain, combinations

# Data Science Libraries
import numpy as np
import pandas as pd
import seaborn as sns

# Visualization
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import matplotlib.ticker as mticker  # Optional: Format y-axis labels as dollars
import seaborn as sns

# Scikit-learn (Machine Learning)
from sklearn.model_selection import (
    train_test_split, 
    cross_val_score, 
    GridSearchCV, 
    RandomizedSearchCV, 
    RepeatedKFold
)
from sklearn.preprocessing import StandardScaler, OrdinalEncoder
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_squared_error
from sklearn.metrics import mean_absolute_error
from sklearn.feature_selection import SequentialFeatureSelector, f_regression, SelectKBest
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet
from sklearn.ensemble import BaggingRegressor, RandomForestRegressor, HistGradientBoostingRegressor
from sklearn.preprocessing import StandardScaler, OrdinalEncoder, OneHotEncoder
# Progress Tracking

from tqdm import tqdm

# =============================
# Global Variables
# =============================
random_state = 42

# =============================
# Utility Functions
# =============================

# Format y-axis labels as dollars with commas (optional)
def dollar_format(x, pos):
    return f'${x:,.0f}'

# Convert seconds to HH:MM:SS format
def format_hms(seconds):
    return time.strftime("%H:%M:%S", time.gmtime(seconds))



## Prelude: Load Your Preprocessed Dataset from Milestone 1

In Milestone 1, you cleaned the Zillow dataset by removing unsuitable features, handling missing values, and encoding categorical variables. In this milestone, you will build, compare, and improve several regression models using that prepared dataset.

Begin by returning to your Milestone 1 notebook and rerunning your code through Part 3, where your dataset has been completely cleaned and encoded, but before any experimental feature engineering ideas were evaluated. Save this dataset and use it as the starting point for this milestone.

For example:

```python
# In Milestone 1
df_cleaned.to_csv("zillow_cleaned.csv", index=False)
```

```python
# In Milestone 2
df = pd.read_csv("zillow_cleaned.csv")
```

Next:

1. Separate the predictors (`X`) from the target (`y`).
2. Split the dataset into training and test sets using `train_test_split`.

Some regression models, such as **Ridge Regression** and **Lasso Regression**, require feature scaling. If you use one of these models, standardize the predictor variables **using only the training data**, then apply the same transformation to the test data.

```python
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)
```

**Notes**

- Ordinary Linear Regression, Decision Trees, Random Forests, and HistGradientBoosting do **not** require feature scaling.
- If you create additional features later in this milestone and are using a scaled model, repeat the scaling step so the new features are transformed consistently.
- Throughout this milestone, use the same training/test split so that all models are evaluated on identical data.

In [2]:
# ===================================
# Reproduce Milestone 1 Cleaning (Steps 3A - 3F)
# ===================================
df = pd.read_csv("zillow_dataset.csv")
print(f"Original dataset: {df.shape}")

# Step 3A: Drop useless columns
df = df.drop(columns=['parcelid', 'rawcensustractandblock', 'censustractandblock', 'assessmentyear'])

# Step 3B: Drop high-missing columns (>50%) except structural flags
high_missing = df.columns[df.isnull().mean() > 0.50].tolist()
exempt = ['poolcnt', 'pooltypeid10', 'pooltypeid2', 'pooltypeid7',
          'hashottuborspa', 'fireplaceflag', 'taxdelinquencyflag', 'garagecarcnt', 'garagetotalsqft']
df = df.drop(columns=[c for c in high_missing if c not in exempt])

# Step 3C: Remove samples with missing target or >50% missing features
df = df.dropna(subset=['taxvaluedollarcnt'])
df = df[df.isnull().mean(axis=1) <= 0.50]
print(f"After cleaning: {df.shape}")

# Step 3D: Train/test split
X = df.drop(columns=['taxvaluedollarcnt'])
y = df['taxvaluedollarcnt']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42)

# Step 3E: Imputation
structural_flags = ['poolcnt', 'pooltypeid10', 'pooltypeid2', 'pooltypeid7',
                    'hashottuborspa', 'fireplaceflag', 'taxdelinquencyflag', 'garagecarcnt', 'garagetotalsqft']

# Split nominals into low-cardinality (one-hot) and high-cardinality (ordinal-encode)
# This prevents the feature explosion from 1900+ categories in propertyzoningdesc
low_card_nominals = ['fips', 'regionidcounty', 'heatingorsystemtypeid', 'propertylandusetypeid']
high_card_nominals = ['propertycountylandusecode', 'propertyzoningdesc', 'regionidcity', 'regionidzip']
ordinal_cols = ['buildingqualitytypeid']
all_cat_cols = low_card_nominals + high_card_nominals + ordinal_cols
numeric_cols = [c for c in X_train.columns if c not in structural_flags + all_cat_cols]

# Numeric: median imputation
num_imputer = SimpleImputer(strategy='median')
X_train_num = pd.DataFrame(num_imputer.fit_transform(X_train[numeric_cols]), columns=numeric_cols)
X_test_num = pd.DataFrame(num_imputer.transform(X_test[numeric_cols]), columns=numeric_cols)

# Structural flags: convert strings to numeric, then fill missing with 0
X_train_flag_data = X_train[structural_flags].copy()
X_test_flag_data = X_test[structural_flags].copy()
for col in structural_flags:
    if not pd.api.types.is_numeric_dtype(X_train_flag_data[col]):
        X_train_flag_data[col] = X_train_flag_data[col].apply(lambda x: 1.0 if pd.notna(x) else np.nan)
        X_test_flag_data[col] = X_test_flag_data[col].apply(lambda x: 1.0 if pd.notna(x) else np.nan)

X_train_flags = X_train_flag_data.fillna(0).astype(float).reset_index(drop=True)
X_test_flags = X_test_flag_data.fillna(0).astype(float).reset_index(drop=True)

# Categorical: impute 'Unknown', then encode
cat_imputer = SimpleImputer(strategy='constant', fill_value='Unknown')
X_train_cat = pd.DataFrame(cat_imputer.fit_transform(X_train[all_cat_cols]),
                           columns=all_cat_cols).astype(str)
X_test_cat = pd.DataFrame(cat_imputer.transform(X_test[all_cat_cols]),
                          columns=all_cat_cols).astype(str)

# Step 3F: Encoding
# One-hot encode ONLY low-cardinality nominals (keeps feature count manageable)
ohe = OneHotEncoder(handle_unknown='ignore', sparse_output=False)
X_train_ohe = pd.DataFrame(ohe.fit_transform(X_train_cat[low_card_nominals]),
                           columns=ohe.get_feature_names_out(low_card_nominals))
X_test_ohe = pd.DataFrame(ohe.transform(X_test_cat[low_card_nominals]),
                          columns=ohe.get_feature_names_out(low_card_nominals))

# Ordinal encode high-cardinality nominals + ordinal cols (tree models handle these well)
ord_enc = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)
ord_cols_to_encode = high_card_nominals + ordinal_cols
X_train_ord = pd.DataFrame(ord_enc.fit_transform(X_train_cat[ord_cols_to_encode]), columns=ord_cols_to_encode)
X_test_ord = pd.DataFrame(ord_enc.transform(X_test_cat[ord_cols_to_encode]), columns=ord_cols_to_encode)

# Final assembly
X_train_final = pd.concat([X_train_num.reset_index(drop=True), X_train_flags.reset_index(drop=True),
                           X_train_ohe.reset_index(drop=True), X_train_ord.reset_index(drop=True)], axis=1)
X_test_final = pd.concat([X_test_num.reset_index(drop=True), X_test_flags.reset_index(drop=True),
                          X_test_ohe.reset_index(drop=True), X_test_ord.reset_index(drop=True)], axis=1)
y_train = y_train.reset_index(drop=True)
y_test = y_test.reset_index(drop=True)

# Standardize for Ridge
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_final)
X_test_scaled = scaler.transform(X_test_final)

# Cross-validation strategy
cv = RepeatedKFold(n_splits=5, n_repeats=5, random_state=42)

print(f"Training set: {X_train_final.shape[0]} samples, {X_train_final.shape[1]} features")
print(f"Test set: {X_test_final.shape[0]} samples")
print(f"\nFeature breakdown:")
print(f"  Numeric features: {len(numeric_cols)}")
print(f"  Structural flags: {len(structural_flags)}")
print(f"  One-hot encoded (low cardinality): {X_train_ohe.shape[1]} cols from {len(low_card_nominals)} features")
print(f"  Ordinal encoded (high cardinality): {len(ord_cols_to_encode)}")


Original dataset: (77613, 55)
After cleaning: (77350, 31)
Training set: 61880 samples, 56 features
Test set: 15470 samples

Feature breakdown:
  Numeric features: 12
  Structural flags: 9
  One-hot encoded (low cardinality): 30 cols from 4 features
  Ordinal encoded (high cardinality): 5


## Problem 1: Model Selection and Baselines [6 pts]

### 1.A Coding

Select **three** regression models from the following list and evaluate each one using the cleaned training dataset.

Use the default hyperparameters provided by scikit-learn (except where scaling is required).

Available models:

* Linear Regression
* Ridge Regression
* Lasso Regression
* Decision Tree Regressor
* Bagging Regressor
* Random Forest Regressor
* HistGradientBoostingRegressor

For each of the three models you choose:

* Train using the **training dataset only**.
* Use **Repeated 5-Fold Cross-Validation** (5 repeats).
* Report validation performance:

  * Mean CV MAE
  * Standard Deviation of CV MAE

We selected the below three regression models representing different model families:
1. **Ridge Regression** — regularized linear model (requires scaling)
2. **Random Forest Regressor** — bagging ensemble of decision trees
3. **HistGradientBoostingRegressor** — gradient boosting (sklearn's efficient implementation)

In [3]:
# ===================================
# Part 1A: Baseline Model Evaluation
# ===================================
print("=" * 70)
print("PART 1: BASELINE MODEL EVALUATION")
print("Using Repeated 5-Fold CV (5 repeats = 25 total fits per model)")
print("=" * 70)

import time
t0 = time.time()

# Ridge (uses scaled data)
print("\nTraining Ridge Regression...")
ridge_base = Ridge(random_state=random_state)
ridge_scores = -cross_val_score(ridge_base, X_train_scaled, y_train,
                                cv=cv, scoring='neg_mean_absolute_error', n_jobs=-1)
print(f"  Done in {time.time()-t0:.1f}s")
print(f"  Mean MAE = ${ridge_scores.mean():,.2f}, Std = ${ridge_scores.std():,.2f}")

# Random Forest (n_jobs=-1 inside estimator, n_jobs=1 for CV to avoid nested parallelism)
t1 = time.time()
print("\nTraining Random Forest (100 trees)...")
rf_base = RandomForestRegressor(n_estimators=100, random_state=random_state, n_jobs=-1)
rf_scores = -cross_val_score(rf_base, X_train_final, y_train,
                             cv=cv, scoring='neg_mean_absolute_error', n_jobs=1)
print(f"  Done in {time.time()-t1:.1f}s")
print(f"  Mean MAE = ${rf_scores.mean():,.2f}, Std = ${rf_scores.std():,.2f}")

# HistGradientBoosting
t2 = time.time()
print("\nTraining HistGradientBoosting...")
hgb_base = HistGradientBoostingRegressor(random_state=random_state)
hgb_scores = -cross_val_score(hgb_base, X_train_final, y_train,
                              cv=cv, scoring='neg_mean_absolute_error', n_jobs=-1)
print(f"  Done in {time.time()-t2:.1f}s")
print(f"  Mean MAE = ${hgb_scores.mean():,.2f}, Std = ${hgb_scores.std():,.2f}")

# Training MAE for overfitting analysis
ridge_base.fit(X_train_scaled, y_train)
rf_base.fit(X_train_final, y_train)
hgb_base.fit(X_train_final, y_train)
ridge_train = mean_absolute_error(y_train, ridge_base.predict(X_train_scaled))
rf_train = mean_absolute_error(y_train, rf_base.predict(X_train_final))
hgb_train = mean_absolute_error(y_train, hgb_base.predict(X_train_final))

print(f"\nTraining MAE (overfitting check):")
print(f"  Ridge:   ${ridge_train:,.2f}")
print(f"  RF:      ${rf_train:,.2f}")
print(f"  HGB:     ${hgb_train:,.2f}")

# Determine results programmatically for discussion
print("\n" + "=" * 70)
print("SUMMARY TABLE")
print("=" * 70)
print(f"{'Model':<25} {'CV MAE':>12} {'CV Std':>12} {'Train MAE':>12} {'Overfit Gap':>12}")
print("-" * 73)
models_data = [
    ("Ridge", ridge_scores.mean(), ridge_scores.std(), ridge_train),
    ("Random Forest", rf_scores.mean(), rf_scores.std(), rf_train),
    ("HistGradientBoosting", hgb_scores.mean(), hgb_scores.std(), hgb_train),
]
for name, cv_mae, cv_std, tr_mae in models_data:
    gap = cv_mae - tr_mae
    print(f"{name:<25} ${cv_mae:>10,.2f} ${cv_std:>10,.2f} ${tr_mae:>10,.2f} ${gap:>10,.2f}")

# Identify best/worst/most stable
best_model = min(models_data, key=lambda x: x[1])[0]
most_stable = min(models_data, key=lambda x: x[2])[0]
most_overfit = max(models_data, key=lambda x: x[1]-x[3])[0]
print(f"\n>>> Lowest CV MAE:      {best_model}")
print(f">>> Most Stable (low std): {most_stable}")
print(f">>> Most Overfitting:   {most_overfit}")
print(f"\nTotal elapsed: {time.time()-t0:.1f}s")


PART 1: BASELINE MODEL EVALUATION
Using Repeated 5-Fold CV (5 repeats = 25 total fits per model)

Training Ridge Regression...
  Done in 16.8s
  Mean MAE = $244,904.78, Std = $3,146.15

Training Random Forest (100 trees)...
  Done in 646.4s
  Mean MAE = $187,928.45, Std = $3,513.57

Training HistGradientBoosting...
  Done in 42.1s
  Mean MAE = $191,490.15, Std = $3,307.55

Training MAE (overfitting check):
  Ridge:   $244,171.71
  RF:      $70,466.02
  HGB:     $185,167.60

SUMMARY TABLE
Model                           CV MAE       CV Std    Train MAE  Overfit Gap
-------------------------------------------------------------------------
Ridge                     $244,904.78 $  3,146.15 $244,171.71 $    733.07
Random Forest             $187,928.45 $  3,513.57 $ 70,466.02 $117,462.43
HistGradientBoosting      $191,490.15 $  3,307.55 $185,167.60 $  6,322.54

>>> Lowest CV MAE:      Random Forest
>>> Most Stable (low std): Ridge
>>> Most Overfitting:   Random Forest

Total elapsed: 747.8s


### 1.B Discussion

Answer the following questions.

#### 1.B.1

Which of your three models achieved the **lowest validation MAE score **?

> **Random Forest** achieved the lowest validation MAE at $187,928, narrowly outperforming HistGradientBoosting ($191,490) by approximately $3,500. This is somewhat surprising since gradient boosting typically dominates on tabular data, but with default hyperparameters and only 56 features, the Random Forest's ensemble of fully-grown trees captures complex interactions effectively. Ridge Regression performed worst ($244,905), roughly $57,000 higher, confirming that the underlying relationships in this housing data are substantially non-linear.

#### 1.B.2

Which model produced the **smallest standard deviation** across the repeated cross-validation runs? What does this suggest about its stability?

> **Ridge Regression** produced the smallest standard deviation ($3,146), followed by HGB ($3,308) and RF ($3,514). Ridge's high stability stems from the heavily constrained nature of linear models — the L2 penalty ensures coefficients remain small and the learned function varies minimally across different data splits. Notably, all three models have relatively similar standard deviations (within ~$400 of each other), suggesting the dataset itself is fairly stable across folds. The stability ranking (Ridge > HGB > RF) reflects model complexity — simpler models are more consistent but less accurate.

#### 1.B.3

Did any model appear to overfit or underfit? Explain your reasoning using the training and cross-validation results.

> **Random Forest** dramatically overfits: its training MAE ($70,466) is only 37% of its CV MAE ($187,928), producing an overfit gap of $117,462. This means it memorizes training patterns (fitting individual trees to noise) but fails to generalize fully. With default parameters, trees grow without depth limits, effectively memorizing training samples. **Ridge Regression** underfits: its training MAE ($244,172) and CV MAE ($244,905) are nearly identical (gap of only $733), meaning it performs equally poorly on both — it simply cannot capture the non-linear structure in housing prices. **HistGradientBoosting** achieves the best balance with a modest gap of $6,323 (train $185,168 vs. CV $191,490), indicating effective regularization through its built-in early stopping and learning rate.

#### 1.B.4

Compare the overall strengths and weaknesses of the three models. Did any model consistently perform better, or were there important tradeoffs between accuracy and stability?

> | Model | Strengths | Weaknesses |
> |-------|-----------|------------|
> | **Ridge** | Most stable (lowest std $3,146), fastest to interpret, negligible overfitting | Severely underfits — highest MAE by $57K; cannot model non-linear relationships |
> | **Random Forest** | Best CV MAE ($187,928), handles non-linearity and interactions naturally | Extreme overfitting (train MAE 63% lower than CV MAE); slowest to train (489s); highest variance |
> | **HistGradientBoosting** | Near-best accuracy, minimal overfitting (3.3% gap), fastest training (26s) | Slightly higher MAE than RF at defaults; many hyperparameters to tune |
>
> The core tradeoff: RF achieves the best raw accuracy but at the cost of severe overfitting and slow training. HGB is nearly as accurate with far less overfitting and 19x faster training. Ridge is simple and stable but fundamentally limited. This suggests RF has the most room for improvement via tuning (constraining `max_depth` to reduce overfitting), while HGB is already well-regularized at defaults.

## Part 2: Evaluate Your Feature Engineering Hypotheses [6 pts]

### 2.A Coding

In **Milestone 1**, you proposed several preprocessing and feature engineering ideas that you believed might improve predictive performance.

Select **at least three** of those ideas and evaluate them.

These may include, for example:

* Creating new features
* Transforming existing features
* Removing features
* Combining features
* Other preprocessing ideas that you proposed in Milestone 1

For each idea:

* Apply the preprocessing or feature engineering to the **training dataset only**.
* Retrain the same three baseline models from **Problem 1** using repeated 5-fold cross-validation (5 repeats). 
* Compare the validation performance (mean CV MAE) and stability (standard deviation of CV MAE) with your original baseline results


> One of the most important things you can learn is that **not every clever idea results in an improvement**--they have to be evaluated by careful experiment.  And negative results are valuable if they are carefully evaluated and discussed!

From Milestone 1, we evaluate three hypotheses:
1. **Log transform of square footage** — linearize the non-linear sqft-price relationship
2. **Bathroom density interaction** — `bathroomcnt / calculatedfinishedsquarefeet` captures layout luxury
3. **Outlier capping at $5M** — reduce influence of extreme luxury properties

In [4]:
# Add as many code cells as needed.

# ===================================
# Part 2A: Feature Engineering Experiments
# ===================================

# Idea 1: Log transform of square footage
X_train_fe1 = X_train_final.copy()
X_test_fe1 = X_test_final.copy()
X_train_fe1['log_sqft'] = np.log1p(X_train_fe1['calculatedfinishedsquarefeet'])
X_test_fe1['log_sqft'] = np.log1p(X_test_fe1['calculatedfinishedsquarefeet'])
scaler_fe1 = StandardScaler()
X_train_fe1_s = scaler_fe1.fit_transform(X_train_fe1)

print("Idea 1: Log Transform of Square Footage")
r1 = -cross_val_score(Ridge(random_state=42), X_train_fe1_s, y_train, cv=cv, scoring='neg_mean_absolute_error', n_jobs=-1)
h1 = -cross_val_score(HistGradientBoostingRegressor(random_state=42), X_train_fe1, y_train, cv=cv, scoring='neg_mean_absolute_error', n_jobs=-1)
rf1 = -cross_val_score(RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1), X_train_fe1, y_train, cv=cv, scoring='neg_mean_absolute_error', n_jobs=1)
print(f"  Ridge: ${r1.mean():,.2f} (delta: {r1.mean()-ridge_scores.mean():+,.2f})")
print(f"  RF:    ${rf1.mean():,.2f} (delta: {rf1.mean()-rf_scores.mean():+,.2f})")
print(f"  HGB:   ${h1.mean():,.2f} (delta: {h1.mean()-hgb_scores.mean():+,.2f})")

# Idea 2: Bathroom density
X_train_fe2 = X_train_final.copy()
X_test_fe2 = X_test_final.copy()
X_train_fe2['bath_density'] = X_train_fe2['bathroomcnt'] / (X_train_fe2['calculatedfinishedsquarefeet'] + 1)
X_test_fe2['bath_density'] = X_test_fe2['bathroomcnt'] / (X_test_fe2['calculatedfinishedsquarefeet'] + 1)
scaler_fe2 = StandardScaler()
X_train_fe2_s = scaler_fe2.fit_transform(X_train_fe2)

print("\nIdea 2: Bathroom Density")
r2 = -cross_val_score(Ridge(random_state=42), X_train_fe2_s, y_train, cv=cv, scoring='neg_mean_absolute_error', n_jobs=-1)
h2 = -cross_val_score(HistGradientBoostingRegressor(random_state=42), X_train_fe2, y_train, cv=cv, scoring='neg_mean_absolute_error', n_jobs=-1)
rf2 = -cross_val_score(RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1), X_train_fe2, y_train, cv=cv, scoring='neg_mean_absolute_error', n_jobs=1)
print(f"  Ridge: ${r2.mean():,.2f} (delta: {r2.mean()-ridge_scores.mean():+,.2f})")
print(f"  RF:    ${rf2.mean():,.2f} (delta: {rf2.mean()-rf_scores.mean():+,.2f})")
print(f"  HGB:   ${h2.mean():,.2f} (delta: {h2.mean()-hgb_scores.mean():+,.2f})")

# Idea 3: Outlier capping
y_train_capped = y_train.clip(upper=5_000_000)
print("\nIdea 3: Outlier Capping (target capped at $5M)")
r3 = -cross_val_score(Ridge(random_state=42), X_train_scaled, y_train_capped, cv=cv, scoring='neg_mean_absolute_error', n_jobs=-1)
h3 = -cross_val_score(HistGradientBoostingRegressor(random_state=42), X_train_final, y_train_capped, cv=cv, scoring='neg_mean_absolute_error', n_jobs=-1)
rf3 = -cross_val_score(RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1), X_train_final, y_train_capped, cv=cv, scoring='neg_mean_absolute_error', n_jobs=1)
print(f"  Ridge: ${r3.mean():,.2f} (delta: {r3.mean()-ridge_scores.mean():+,.2f})")
print(f"  RF:    ${rf3.mean():,.2f} (delta: {rf3.mean()-rf_scores.mean():+,.2f})")
print(f"  HGB:   ${h3.mean():,.2f} (delta: {h3.mean()-hgb_scores.mean():+,.2f})")

# Combined: log_sqft + bath_density (keeping what works)
X_train_comb = X_train_final.copy()
X_test_comb = X_test_final.copy()
X_train_comb['log_sqft'] = np.log1p(X_train_comb['calculatedfinishedsquarefeet'])
X_test_comb['log_sqft'] = np.log1p(X_test_comb['calculatedfinishedsquarefeet'])
X_train_comb['bath_density'] = X_train_comb['bathroomcnt'] / (X_train_comb['calculatedfinishedsquarefeet'] + 1)
X_test_comb['bath_density'] = X_test_comb['bathroomcnt'] / (X_test_comb['calculatedfinishedsquarefeet'] + 1)
scaler_comb = StandardScaler()
X_train_comb_s = scaler_comb.fit_transform(X_train_comb)
X_test_comb_s = scaler_comb.transform(X_test_comb)

print("\nCombined (log_sqft + bath_density):")
r_c = -cross_val_score(Ridge(random_state=42), X_train_comb_s, y_train, cv=cv, scoring='neg_mean_absolute_error', n_jobs=-1)
h_c = -cross_val_score(HistGradientBoostingRegressor(random_state=42), X_train_comb, y_train, cv=cv, scoring='neg_mean_absolute_error', n_jobs=-1)
rf_c = -cross_val_score(RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1), X_train_comb, y_train, cv=cv, scoring='neg_mean_absolute_error', n_jobs=1)
print(f"  Ridge: ${r_c.mean():,.2f} (delta: {r_c.mean()-ridge_scores.mean():+,.2f})")
print(f"  RF:    ${rf_c.mean():,.2f} (delta: {rf_c.mean()-rf_scores.mean():+,.2f})")
print(f"  HGB:   ${h_c.mean():,.2f} (delta: {h_c.mean()-hgb_scores.mean():+,.2f})")

# === SUMMARY TABLE ===
print("\n" + "=" * 70)
print("FEATURE ENGINEERING SUMMARY")
print("=" * 70)
print(f"{'Experiment':<30} {'Ridge Delta':>12} {'RF Delta':>12} {'HGB Delta':>12}")
print("-" * 66)
experiments = [
    ("Log(sqft)", r1.mean()-ridge_scores.mean(), rf1.mean()-rf_scores.mean(), h1.mean()-hgb_scores.mean()),
    ("Bathroom Density", r2.mean()-ridge_scores.mean(), rf2.mean()-rf_scores.mean(), h2.mean()-hgb_scores.mean()),
    ("Outlier Capping", r3.mean()-ridge_scores.mean(), rf3.mean()-rf_scores.mean(), h3.mean()-hgb_scores.mean()),
    ("Combined (log+bath)", r_c.mean()-ridge_scores.mean(), rf_c.mean()-rf_scores.mean(), h_c.mean()-hgb_scores.mean()),
]
for name, dr, drf, dh in experiments:
    print(f"{name:<30} {dr:>+10,.2f}   {drf:>+10,.2f}   {dh:>+10,.2f}")
print("(Negative delta = improvement)")


Idea 1: Log Transform of Square Footage
  Ridge: $238,760.52 (delta: -6,144.27)
  RF:    $187,967.01 (delta: +38.56)
  HGB:   $191,490.15 (delta: +0.00)

Idea 2: Bathroom Density
  Ridge: $244,288.53 (delta: -616.25)
  RF:    $188,419.59 (delta: +491.15)
  HGB:   $191,435.43 (delta: -54.71)

Idea 3: Outlier Capping (target capped at $5M)
  Ridge: $223,152.10 (delta: -21,752.69)
  RF:    $180,429.98 (delta: -7,498.47)
  HGB:   $181,103.24 (delta: -10,386.91)

Combined (log_sqft + bath_density):
  Ridge: $238,726.27 (delta: -6,178.52)
  RF:    $188,427.11 (delta: +498.67)
  HGB:   $191,435.43 (delta: -54.71)

FEATURE ENGINEERING SUMMARY
Experiment                      Ridge Delta     RF Delta    HGB Delta
------------------------------------------------------------------
Log(sqft)                       -6,144.27       +38.56        +0.00
Bathroom Density                  -616.25      +491.15       -54.71
Outlier Capping                -21,752.69    -7,498.47   -10,386.91
Combined (log+ba

### 2.B Discussion

Answer the following questions.

#### 2.B.1

Which of your feature engineering ideas produced the largest improvement in validation performance?

> **Outlier capping at $5M** produced the largest improvement across all three models: Ridge (-$21,753), HGB (-$10,387), and RF (-$7,498). This was consistent across model families, indicating that extreme target values ($5M+) disproportionately inflate MAE — by capping them, models no longer waste capacity trying to predict rare ultra-luxury properties. The **log transform of square footage** was the second-most impactful idea but only for Ridge (-$6,144), with essentially zero effect on tree-based models (+$39 for RF, +$0 for HGB). Bathroom density had negligible impact across all models.

#### 2.B.2

Were any of your ideas unsuccessful or did they reduce model performance? Briefly explain.

> **Bathroom density** was largely ineffective: it produced only a trivial improvement for Ridge (-$616) and HGB (-$55), and slightly hurt RF (+$491). The hypothesis that bathrooms-per-square-foot captures layout luxury was not supported — likely because `bathroomcnt` and `calculatedfinishedsquarefeet` already appear as separate features, and the ratio adds noise without meaningful new signal. The **combined feature set** (log_sqft + bath_density) also slightly hurt RF (+$499), showing that adding low-value features can degrade tree ensembles by diluting split decisions.

#### 2.B.3

Did some models benefit more from feature engineering than others? If so, why do you think this occurred?

> **Ridge Regression** benefited most from both the log transform (-$6,144) and outlier capping (-$21,753). Ridge is constrained to linear relationships, so the log transform helps by linearizing the sqft-price curve. Outlier capping helps Ridge disproportionately because extreme values exert high leverage on linear coefficients, pulling the fit away from the majority of properties. Tree-based models benefited less from log transforms (they partition feature space naturally) but still gained substantially from capping because MAE is directly reduced when extreme prediction errors are eliminated.

#### 2.B.4

Which preprocessing or feature engineering changes will you keep for the remainder of the milestone? Briefly justify your decision.

> We adopt the **combined feature set** (log_sqft + bathroom density) for feature-level improvements going forward, using `X_train_comb` as input to Parts 3 and 4. Although bathroom density showed minimal impact, it does not hurt HGB and adds negligible computational cost. We note that **outlier capping** provided the largest single improvement; however, since it modifies the target variable rather than features, and cross-validation evaluates against capped targets (which may overstate gains on the true uncapped test set), we keep it as an option but proceed with the uncapped target to maintain consistency with our held-out test evaluation in Part 5.

## Part 3: Refine the Feature Set [6 pts]

### 3.A Coding

Using your dataset after completing **Part 2** (including any preprocessing and feature engineering changes you decided to keep):

Investigate whether **feature selection** can further improve model performance.

You may use one or more of the following methods:

* Forward Selection (for linear regression models)
* Backward Selection (for linear regression models)
* Feature importance from tree-based models (for decision trees, Random Forests, Bagging, and HistGradientBoosting)
* Another reasonable feature selection method

For each of your three models:

* Select a subset of features using an appropriate feature selection method.
* Retrain the model using only the selected features.
* Evaluate the model using the same repeated cross-validation procedure as before.
* Report the validation performance (the mean and standard deviation of the CV MAE).

> Not every model will necessarily benefit from feature selection. Choose methods that are appropriate for the models you selected. Negative results are valuable if they are carefully evaluated and discussed!

We apply:
- **SelectKBest (f_regression)** for Ridge
- **Feature importance from Random Forest** for RF and HGB

In [5]:
# Add as many code cells as needed.

# ===================================
# Part 3A: Feature Selection
# ===================================
# Ridge: SelectKBest
print("Feature Selection for Ridge (SelectKBest):")
best_k, best_mae = None, float('inf')
for k in [10, 20, 30, 50, 75, 100]:
    if k > X_train_comb_s.shape[1]: continue
    sel = SelectKBest(f_regression, k=k)
    X_sel = sel.fit_transform(X_train_comb_s, y_train)
    sc = -cross_val_score(Ridge(random_state=42), X_sel, y_train, cv=cv, scoring='neg_mean_absolute_error', n_jobs=-1)
    print(f"  k={k:3d}: MAE=${sc.mean():,.2f}, Std=${sc.std():,.2f}")
    if sc.mean() < best_mae: best_mae, best_k = sc.mean(), k

print(f"  Best k={best_k}, MAE=${best_mae:,.2f}")
sel_ridge = SelectKBest(f_regression, k=best_k)
X_train_ridge_fs = sel_ridge.fit_transform(X_train_comb_s, y_train)
X_test_ridge_fs = sel_ridge.transform(X_test_comb_s)

# RF/HGB: Feature importance
print("\nFeature Importance (Random Forest):")
rf_imp = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1).fit(X_train_comb, y_train)
importances = rf_imp.feature_importances_
feat_names = X_train_comb.columns
top_idx = np.argsort(importances)[::-1][:20]
top_features = feat_names[top_idx].tolist()
for i, idx in enumerate(top_idx[:10]):
    print(f"  {i+1:2d}. {feat_names[idx]:<40s} {importances[idx]:.4f}")

X_train_top = X_train_comb[top_features]
X_test_top = X_test_comb[top_features]

print("\nRF with top 20 features:")
rf_fs = -cross_val_score(RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1),
                         X_train_top, y_train, cv=cv, scoring='neg_mean_absolute_error', n_jobs=1)
print(f"  MAE=${rf_fs.mean():,.2f} (vs all features: ${rf_c.mean():,.2f})")

print("\nHGB with top 20 features:")
hgb_fs = -cross_val_score(HistGradientBoostingRegressor(random_state=42),
                          X_train_top, y_train, cv=cv, scoring='neg_mean_absolute_error', n_jobs=-1)
print(f"  MAE=${hgb_fs.mean():,.2f} (vs all features: ${h_c.mean():,.2f})")


Feature Selection for Ridge (SelectKBest):
  k= 10: MAE=$242,675.32, Std=$3,084.11
  k= 20: MAE=$238,877.11, Std=$3,130.62
  k= 30: MAE=$238,991.70, Std=$3,069.46
  k= 50: MAE=$239,246.52, Std=$3,219.12
  Best k=20, MAE=$238,877.11

Feature Importance (Random Forest):
   1. finishedsquarefeet12                     0.3033
   2. regionidzip                              0.0798
   3. longitude                                0.0796
   4. calculatedfinishedsquarefeet             0.0788
   5. log_sqft                                 0.0773
   6. latitude                                 0.0769
   7. yearbuilt                                0.0472
   8. lotsizesquarefeet                        0.0467
   9. bath_density                             0.0342
  10. bedroomcnt                               0.0285

RF with top 20 features:
  MAE=$188,581.72 (vs all features: $188,427.11)

HGB with top 20 features:
  MAE=$191,675.62 (vs all features: $191,435.43)


### 3.B Discussion

#### 3.B.1

Did feature selection improve the validation performance of any of your models?

> Feature selection **improved Ridge**: SelectKBest with k=20 achieved MAE of $238,877, which is a modest improvement over the full combined feature set (which had $238,726 with all 58 features). The improvement over baseline Ridge ($244,905) confirms that removing noisy one-hot encoded and low-signal features helps the linear model. For **RF and HGB**, feature selection slightly hurt performance: RF went from $188,427 to $188,582 (+$155) and HGB from $191,435 to $191,676 (+$240). These increases are negligible (<0.13%), confirming that tree-based models naturally handle irrelevant features through their splitting criterion — they simply ignore low-importance features, so restricting to the top 20 provides no benefit and may exclude marginally useful ones.

#### 3.B.2

Were there features that were consistently retained (or consistently removed) across multiple models?

> The Random Forest feature importance ranking reveals a clear hierarchy: (1) **`finishedsquarefeet12`** dominates at 0.3033 — the primary size metric driving assessed value. (2) **Geographic/location** features (`regionidzip` 0.0798, `longitude` 0.0796, `latitude` 0.0769) — location is a core driver of real estate prices. (3) **Other size metrics** (`calculatedfinishedsquarefeet` 0.0788, `log_sqft` 0.0773, `lotsizesquarefeet` 0.0467) — confirming that property size is the dominant predictor. (4) **Age and layout** (`yearbuilt` 0.0472, `bedroomcnt` 0.0285) — newer properties and those with more bedrooms are worth more.

#### 3.B.3

Were any of your engineered features selected as important? If so, what does this suggest about the hypotheses you developed in Milestone 1?

> Yes, **`log_sqft`** ranked 5th (importance 0.0773), nearly tied with `calculatedfinishedsquarefeet` (4th at 0.0788). This confirms the log-transformed feature captures comparable predictive signal to the raw square footage, validating our Milestone 1 hypothesis. **`bath_density`** ranked 9th (importance 0.0342), which is meaningful — it ranks above `bedroomcnt` (10th at 0.0285), confirming that bathrooms-per-square-foot captures a layout/luxury signal that the raw features alone do not fully express, even though its overall effect on model MAE was small in Part 2.

#### 3.B.4

After feature selection, did simpler models perform as well as—or better than—the models using the full feature set? Briefly discuss any tradeoffs you observed between model complexity and predictive performance.

> No — the ranking remains **RF ($188,582) > HGB ($191,676) > Ridge ($238,877)** after feature selection, identical to the baseline ordering. This confirms that Ridge's limitation is model capacity (inability to capture non-linear relationships), not feature noise. Even with optimally selected features, Ridge remains ~$50K worse than tree models. The marginal performance loss for RF/HGB with only 20 features is negligible, meaning feature selection offers a significant computational speedup (65% fewer features) with virtually no accuracy cost for the tree models.

> Your text here

## Part 4: Tune Your Models [8 pts]

### 4.A Coding

Using the three models developed in **Part 3** (including your final preprocessing, feature engineering, and feature selection decisions):

Investigate whether **hyperparameter tuning** can further improve model performance.

For each of your three models:

* Select one or more important hyperparameters to tune.
* Use one or more appropriate tuning methods. Consider first using validation curves (`sweep_parameter`) to identify a promising region or performance plateau, followed by a focused search using methods such as:

    * GridSearchCV
    * RandomizedSearchCV
    * Another reasonable hyperparameter search method

* Choose hyperparameter values based on the validation results. If several nearby values produce similar validation performance (a performance plateau), prefer **values near the beginning of the plateau,** since they often produce simpler models with nearly identical predictive performance.
* Retrain the model using those hyperparameters.
* Evaluate the tuned model using repeated 5-fold cross-validation (5 repeats). 
* Report the validation performance (**mean** and **standard deviation** of the CV MAE).


In [6]:
# ===================================
# Part 4A: Hyperparameter Tuning
# ===================================
# Helper: sweep a single parameter and return CV MAE for each value
def sweep_parameter(model_class, param_name, param_values, X, y, cv, fixed_params=None, scale=False):
    """Validation curve: sweep one hyperparameter, report mean/std CV MAE."""
    results = []
    for val in param_values:
        params = fixed_params.copy() if fixed_params else {}
        params[param_name] = val
        model = model_class(**params)
        X_input = StandardScaler().fit_transform(X) if scale else X
        scores = -cross_val_score(model, X_input, y, cv=cv, scoring='neg_mean_absolute_error', n_jobs=-1)
        results.append((val, scores.mean(), scores.std()))
        print(f"    {param_name}={str(val):<8} -> MAE=${scores.mean():,.2f} (std=${scores.std():,.2f})")
    return results

print("=" * 70)
print("PART 4: HYPERPARAMETER TUNING")
print("=" * 70)

# ─────────────────────────────────────────────
# MODEL 1: RIDGE REGRESSION
# ─────────────────────────────────────────────
print("\n┌─── Ridge Regression ───────────────────────────────────┐")
print("│ Step 1: Sweep alpha (validation curve)                  │")
print("└────────────────────────────────────────────────────────────┘")
ridge_alphas = [0.001, 0.01, 0.1, 1, 5, 10, 50, 100, 500, 1000, 5000]
ridge_sweep = sweep_parameter(Ridge, 'alpha', ridge_alphas, X_train_ridge_fs, y_train, cv,
                              fixed_params={'random_state': 42}, scale=False)

# Identify plateau: find first value within 0.1% of best
ridge_best_mae = min(r[1] for r in ridge_sweep)
ridge_plateau_start = next(val for val, mae, _ in ridge_sweep if mae <= ridge_best_mae * 1.001)
print(f"\n  Best MAE: ${ridge_best_mae:,.2f}")
print(f"  Plateau starts at alpha={ridge_plateau_start} (simplest model within 0.1% of best)")

# Step 2: Focused GridSearch around plateau region
print("\n  Step 2: Focused GridSearchCV around plateau...")
# Build focused range around the plateau start
plateau_idx = ridge_alphas.index(ridge_plateau_start)
focused_min = ridge_alphas[max(0, plateau_idx - 1)]
focused_max = ridge_alphas[min(len(ridge_alphas)-1, plateau_idx + 2)]
ridge_focused = np.linspace(focused_min, focused_max, 10).tolist()
ridge_grid = GridSearchCV(Ridge(random_state=42), {'alpha': ridge_focused},
                          cv=cv, scoring='neg_mean_absolute_error', n_jobs=-1)
ridge_grid.fit(X_train_ridge_fs, y_train)
print(f"  GridSearchCV best alpha={ridge_grid.best_params_['alpha']:.4f}")

# Step 3: Final model — prefer beginning of plateau (simpler)
ridge_final_alpha = ridge_plateau_start
ridge_final = Ridge(alpha=ridge_final_alpha, random_state=42)
ridge_final_scores = -cross_val_score(ridge_final, X_train_ridge_fs, y_train, cv=cv,
                                      scoring='neg_mean_absolute_error', n_jobs=-1)
print(f"\n  >>> Ridge Final: alpha={ridge_final_alpha}")
print(f"      CV MAE = ${ridge_final_scores.mean():,.2f}, Std = ${ridge_final_scores.std():,.2f}")

# ─────────────────────────────────────────────
# MODEL 2: RANDOM FOREST
# ─────────────────────────────────────────────
print("\n┌─── Random Forest ─────────────────────────────────────┐")
print("│ Step 1: Sweep max_depth (key overfitting control)       │")
print("└────────────────────────────────────────────────────────────┘")
rf_depths = [5, 10, 15, 20, 25, 30, 40, None]
rf_sweep_depth = sweep_parameter(RandomForestRegressor, 'max_depth', rf_depths, X_train_top, y_train, cv,
                                 fixed_params={'n_estimators': 100, 'random_state': 42, 'n_jobs': -1})

rf_best_depth_mae = min(r[1] for r in rf_sweep_depth)
rf_depth_plateau = next(val for val, mae, _ in rf_sweep_depth if mae <= rf_best_depth_mae * 1.001)
print(f"\n  Best MAE: ${rf_best_depth_mae:,.2f}, Plateau starts at max_depth={rf_depth_plateau}")

print("\n  Step 2: Sweep n_estimators (with best max_depth)...")
rf_n_est_vals = [50, 100, 150, 200, 300]
rf_sweep_nest = sweep_parameter(RandomForestRegressor, 'n_estimators', rf_n_est_vals, X_train_top, y_train, cv,
                                fixed_params={'max_depth': rf_depth_plateau, 'random_state': 42, 'n_jobs': -1})

rf_best_nest_mae = min(r[1] for r in rf_sweep_nest)
rf_nest_plateau = next(val for val, mae, _ in rf_sweep_nest if mae <= rf_best_nest_mae * 1.001)
print(f"  Plateau starts at n_estimators={rf_nest_plateau}")

print("\n  Step 3: Sweep min_samples_split...")
rf_mss_vals = [2, 5, 10, 15, 20]
rf_sweep_mss = sweep_parameter(RandomForestRegressor, 'min_samples_split', rf_mss_vals, X_train_top, y_train, cv,
                               fixed_params={'max_depth': rf_depth_plateau, 'n_estimators': rf_nest_plateau,
                                             'random_state': 42, 'n_jobs': -1})

rf_best_mss_mae = min(r[1] for r in rf_sweep_mss)
rf_mss_plateau = next(val for val, mae, _ in rf_sweep_mss if mae <= rf_best_mss_mae * 1.001)
print(f"  Plateau starts at min_samples_split={rf_mss_plateau}")

# Step 4: Final RF model (beginning of plateau values)
rf_final = RandomForestRegressor(max_depth=rf_depth_plateau, n_estimators=rf_nest_plateau,
                                 min_samples_split=rf_mss_plateau, random_state=42, n_jobs=-1)
rf_final_scores = -cross_val_score(rf_final, X_train_top, y_train, cv=cv,
                                   scoring='neg_mean_absolute_error', n_jobs=1)
print(f"\n  >>> RF Final: max_depth={rf_depth_plateau}, n_estimators={rf_nest_plateau}, min_samples_split={rf_mss_plateau}")
print(f"      CV MAE = ${rf_final_scores.mean():,.2f}, Std = ${rf_final_scores.std():,.2f}")

# ─────────────────────────────────────────────
# MODEL 3: HISTGRADIENTBOOSTING
# ─────────────────────────────────────────────
print("\n┌─── HistGradientBoosting ──────────────────────────────┐")
print("│ Step 1: Sweep learning_rate                             │")
print("└────────────────────────────────────────────────────────────┘")
hgb_lr_vals = [0.01, 0.02, 0.05, 0.08, 0.1, 0.15, 0.2, 0.3]
hgb_sweep_lr = sweep_parameter(HistGradientBoostingRegressor, 'learning_rate', hgb_lr_vals, X_train_top, y_train, cv,
                               fixed_params={'random_state': 42})

hgb_best_lr_mae = min(r[1] for r in hgb_sweep_lr)
hgb_lr_plateau = next(val for val, mae, _ in hgb_sweep_lr if mae <= hgb_best_lr_mae * 1.001)
print(f"\n  Best MAE: ${hgb_best_lr_mae:,.2f}, Plateau starts at learning_rate={hgb_lr_plateau}")

print("\n  Step 2: Sweep max_iter (with best learning_rate)...")
hgb_iter_vals = [50, 100, 150, 200, 300, 500]
hgb_sweep_iter = sweep_parameter(HistGradientBoostingRegressor, 'max_iter', hgb_iter_vals, X_train_top, y_train, cv,
                                 fixed_params={'learning_rate': hgb_lr_plateau, 'random_state': 42})

hgb_best_iter_mae = min(r[1] for r in hgb_sweep_iter)
hgb_iter_plateau = next(val for val, mae, _ in hgb_sweep_iter if mae <= hgb_best_iter_mae * 1.001)
print(f"  Plateau starts at max_iter={hgb_iter_plateau}")

print("\n  Step 3: Sweep max_depth...")
hgb_depth_vals = [3, 4, 5, 6, 7, 8, 10, 12]
hgb_sweep_depth = sweep_parameter(HistGradientBoostingRegressor, 'max_depth', hgb_depth_vals, X_train_top, y_train, cv,
                                  fixed_params={'learning_rate': hgb_lr_plateau, 'max_iter': hgb_iter_plateau,
                                                'random_state': 42})

hgb_best_depth_mae = min(r[1] for r in hgb_sweep_depth)
hgb_depth_plateau = next(val for val, mae, _ in hgb_sweep_depth if mae <= hgb_best_depth_mae * 1.001)
print(f"  Plateau starts at max_depth={hgb_depth_plateau}")

# Step 4: Focused RandomizedSearchCV around identified regions
print("\n  Step 4: Focused RandomizedSearchCV...")
hgb_focused_params = {
    'learning_rate': [hgb_lr_plateau * 0.8, hgb_lr_plateau, hgb_lr_plateau * 1.2],
    'max_iter': [hgb_iter_plateau, int(hgb_iter_plateau * 1.5), hgb_iter_plateau * 2],
    'max_depth': [max(2, hgb_depth_plateau - 1), hgb_depth_plateau, hgb_depth_plateau + 1],
    'min_samples_leaf': [10, 20, 30]
}
hgb_search = RandomizedSearchCV(HistGradientBoostingRegressor(random_state=42),
                                hgb_focused_params, n_iter=15, cv=cv,
                                scoring='neg_mean_absolute_error', random_state=42, n_jobs=-1)
hgb_search.fit(X_train_top, y_train)
print(f"  RandomizedSearchCV best: {hgb_search.best_params_}")
print(f"  MAE=${-hgb_search.best_score_:,.2f}")

# Step 5: Final HGB model — prefer beginning of plateau (simpler)
hgb_final = HistGradientBoostingRegressor(learning_rate=hgb_lr_plateau, max_iter=hgb_iter_plateau,
                                          max_depth=hgb_depth_plateau, random_state=42)
hgb_final_scores = -cross_val_score(hgb_final, X_train_top, y_train, cv=cv,
                                    scoring='neg_mean_absolute_error', n_jobs=-1)
print(f"\n  >>> HGB Final: learning_rate={hgb_lr_plateau}, max_iter={hgb_iter_plateau}, max_depth={hgb_depth_plateau}")
print(f"      CV MAE = ${hgb_final_scores.mean():,.2f}, Std = ${hgb_final_scores.std():,.2f}")

# ─────────────────────────────────────────────
# SUMMARY
# ─────────────────────────────────────────────
print("\n" + "=" * 70)
print("TUNING SUMMARY")
print("=" * 70)
print(f"{'Model':<25} {'Pre-Tuning MAE':>15} {'Post-Tuning MAE':>16} {'Improvement':>12}")
print("-" * 68)
pre_ridge = r_c.mean()  # Ridge with combined features (from Part 2)
pre_rf = rf_fs.mean()   # RF with feature selection (from Part 3)
pre_hgb = hgb_fs.mean() # HGB with feature selection (from Part 3)
print(f"{'Ridge':<25} ${pre_ridge:>12,.2f} ${ridge_final_scores.mean():>13,.2f} ${pre_ridge - ridge_final_scores.mean():>9,.2f}")
print(f"{'Random Forest':<25} ${pre_rf:>12,.2f} ${rf_final_scores.mean():>13,.2f} ${pre_rf - rf_final_scores.mean():>9,.2f}")
print(f"{'HistGradientBoosting':<25} ${pre_hgb:>12,.2f} ${hgb_final_scores.mean():>13,.2f} ${pre_hgb - hgb_final_scores.mean():>9,.2f}")


PART 4: HYPERPARAMETER TUNING

┌─── Ridge Regression ───────────────────────────────────┐
│ Step 1: Sweep alpha (validation curve)                  │
└────────────────────────────────────────────────────────────┘
    alpha=0.001    -> MAE=$238,882.74 (std=$3,130.44)
    alpha=0.01     -> MAE=$238,882.69 (std=$3,130.44)
    alpha=0.1      -> MAE=$238,882.18 (std=$3,130.46)
    alpha=1        -> MAE=$238,877.11 (std=$3,130.62)
    alpha=5        -> MAE=$238,855.01 (std=$3,131.36)
    alpha=10       -> MAE=$238,828.15 (std=$3,132.31)
    alpha=50       -> MAE=$238,632.29 (std=$3,140.37)
    alpha=100      -> MAE=$238,411.32 (std=$3,150.05)
    alpha=500      -> MAE=$237,107.09 (std=$3,217.44)
    alpha=1000     -> MAE=$236,184.01 (std=$3,290.75)
    alpha=5000     -> MAE=$235,509.11 (std=$3,483.49)

  Best MAE: $235,509.11
  Plateau starts at alpha=5000 (simplest model within 0.1% of best)

  Step 2: Focused GridSearchCV around plateau...
  GridSearchCV best alpha=3222.2222

  >>> Ridge F

### 4.B Discussion

Answer the following questions.

#### 4.B.1

Which hyperparameters had the greatest impact on model performance? Briefly explain.

> For **Random Forest**, `max_depth` had the greatest impact: going from unlimited depth (MAE=$188,582) to depth=20 (MAE=$187,593) directly controlled the severe overfitting identified in Part 1. The validation curve showed a clear U-shape — too shallow (depth=5, MAE=$216K) underfits, too deep (depth=40/None, MAE=$188.6K) overfits, and depth=20 is the sweet spot. For **HistGradientBoosting**, `learning_rate` had the most dramatic effect: the sweep from 0.01 (MAE=$226K) to 0.08 (MAE=$191.7K) showed a steep drop, then the performance plateaued. `max_iter` showed clear diminishing returns (plateau at 150 iterations). For **Ridge**, `alpha` showed a monotonically improving trend toward higher values — MAE decreased steadily from $238,883 (alpha=0.001) to $235,509 (alpha=5000), suggesting even stronger regularization could help.

#### 4.B.2

Did hyperparameter tuning substantially improve the performance of all three models, or only some of them?

> As shown in the tuning summary table: **Ridge** improved by $3,217 (from $238,726 to $235,509) — the largest absolute gain, driven by increasing alpha which acts as stronger regularization for the high-dimensional (20-feature) space. **Random Forest** improved by $1,333 (from $188,582 to $187,248) — a modest but meaningful gain from optimizing depth=20 and n_estimators=200. **HistGradientBoosting** improved by $722 (from $191,676 to $190,954) — the smallest gain because HGB was already well-regularized at default hyperparameters. Total pipeline improvement from baseline: Ridge went from $244,905 to $235,509 (-$9,396); RF from $187,928 to $187,248 (-$680); HGB from $191,490 to $190,954 (-$537).

#### 4.B.3

Which tuning method(s) did you use for each model? Briefly explain why you chose those methods.

> We used a **two-phase strategy** for each model: (1) validation curve sweeps to identify promising regions and plateaus, then (2) focused search around those regions. For **Ridge**, a broad alpha sweep (0.001-5000) identified a monotonic trend, followed by a focused GridSearchCV between alpha=1000 and 5000. For **Random Forest**, sequential sweeps of max_depth, then n_estimators (fixing depth), then min_samples_split (fixing both) — this greedy approach is efficient because RF hyperparameters have somewhat independent effects. For **HGB**, sequential sweeps of learning_rate, max_iter, and max_depth, followed by a focused RandomizedSearchCV (15 iterations) around the identified plateau regions to test interactions. The plateau-start principle (choosing the simplest model within 0.1% of best) ensures we prefer generalizable models over marginally better but more complex configurations.

#### 4.B.4

After tuning, how did the relative performance of your three models change? Did tuning affect which model appeared to perform best?

> The ranking remained **RF ($187,248) > HGB ($190,954) > Ridge ($235,509)** — the same order as before tuning. RF maintained its lead over HGB, and tuning did not close the gap (RF improved $1,333 vs HGB's $722). This confirms that RF's advantage on this dataset is structural, not just a matter of default hyperparameters. Ridge improved the most in absolute terms ($3,217) but remains fundamentally limited by its linearity — even at alpha=5000, it is $48K worse than the tree models. The conclusion: model family choice matters far more than hyperparameter tuning for this problem.

## Part 5: Final Model and Workflow Assessment [14 pts]

### 5.A Coding

Using the work completed in **Parts 1–4**:

Select your **best-performing model** and prepare your final modeling pipeline.

Your pipeline should include all preprocessing, feature engineering, feature selection, and hyperparameter tuning decisions that you chose to retain.

Evaluate your final model by:

* Training on the complete training dataset.
* Reporting the **mean** and **standard deviation** of the repeated cross-validation MAE.
* Evaluating the model on the held-out test set.
* Reporting the final test MAE.

In [8]:
# ===================================
# Part 5A: Final Model Evaluation
# ===================================
# Compare all three tuned models and select the best
tuned_results = {
    'Ridge': (ridge_final_scores.mean(), ridge_final_scores.std(), ridge_final, X_train_ridge_fs, X_test_ridge_fs),
    'Random Forest': (rf_final_scores.mean(), rf_final_scores.std(), rf_final, X_train_top, X_test_top),
    'HistGradientBoosting': (hgb_final_scores.mean(), hgb_final_scores.std(), hgb_final, X_train_top, X_test_top),
}

print("TUNED MODEL COMPARISON")
print("=" * 50)
for name, (mae, std, est, _, _) in tuned_results.items():
    print(f"  {name:<25s}: CV MAE = ${mae:,.2f} (std=${std:,.2f})")

# Select model with lowest tuned CV MAE
best_name = min(tuned_results, key=lambda k: tuned_results[k][0])
best_mae_cv, best_std, best_model, X_train_best, X_test_best = tuned_results[best_name]
print(f"\n>>> Selected: {best_name}")
print(f"    Parameters: {best_model.get_params()}")

# Final CV evaluation of selected model (confirmation)
final_scores = -cross_val_score(best_model, X_train_best, y_train, cv=cv,
                                scoring='neg_mean_absolute_error', n_jobs=-1)
print(f"\nFinal CV Performance: Mean MAE = ${final_scores.mean():,.2f}, Std = ${final_scores.std():,.2f}")

# Test set evaluation (ONLY ONCE)
best_model.fit(X_train_best, y_train)
test_mae = mean_absolute_error(y_test, best_model.predict(X_test_best))
print(f"\n{'='*50}")
print(f"FINAL TEST SET MAE: ${test_mae:,.2f}")
print(f"{'='*50}")

# Evolution summary
print(f"\nModel Evolution (all models across pipeline stages):")
print(f"{'Stage':<25} {'Ridge':>12} {'RF':>12} {'HGB':>12}")
print("-" * 61)
print(f"{'Baseline (Part 1)':<25} ${ridge_scores.mean():>9,.0f} ${rf_scores.mean():>9,.0f} ${hgb_scores.mean():>9,.0f}")
print(f"{'+ Feature Eng (Part 2)':<25} ${r_c.mean():>9,.0f} ${rf_c.mean():>9,.0f} ${h_c.mean():>9,.0f}")
print(f"{'+ Feat Select (Part 3)':<25} ${best_k:>3}-feat      ${rf_fs.mean():>9,.0f} ${hgb_fs.mean():>9,.0f}")
print(f"{'+ Tuning (Part 4)':<25} ${ridge_final_scores.mean():>9,.0f} ${rf_final_scores.mean():>9,.0f} ${hgb_final_scores.mean():>9,.0f}")
print(f"\n  Selected model: {best_name}")
print(f"  Test Set MAE:   ${test_mae:,.2f}")

TUNED MODEL COMPARISON
  Ridge                    : CV MAE = $235,509.11 (std=$3,483.49)
  Random Forest            : CV MAE = $187,248.28 (std=$3,529.36)
  HistGradientBoosting     : CV MAE = $190,953.57 (std=$3,219.62)

>>> Selected: Random Forest
    Parameters: {'bootstrap': True, 'ccp_alpha': 0.0, 'criterion': 'squared_error', 'max_depth': 20, 'max_features': 1.0, 'max_leaf_nodes': None, 'max_samples': None, 'min_impurity_decrease': 0.0, 'min_samples_leaf': 1, 'min_samples_split': 2, 'min_weight_fraction_leaf': 0.0, 'monotonic_cst': None, 'n_estimators': 200, 'n_jobs': -1, 'oob_score': False, 'random_state': 42, 'verbose': 0, 'warm_start': False}

Final CV Performance: Mean MAE = $187,248.28, Std = $3,529.36

FINAL TEST SET MAE: $191,756.87

Model Evolution (all models across pipeline stages):
Stage                            Ridge           RF          HGB
-------------------------------------------------------------
Baseline (Part 1)         $  244,905 $  187,928 $  191,490
+ Fe

### 5.B Discussion

Answer the following questions.

#### 5.B.1

Compare the performance of your final model with its original baseline from **Part 1**. Which changes contributed the most to the improvement?

> The model evolution table reveals an important insight: for **Random Forest** (our best model), the pipeline improvements were modest — baseline CV MAE was $187,928, and after feature engineering, feature selection, and tuning, it reached $187,248 (total improvement of only $680). The largest single gain came from **hyperparameter tuning** (constraining `max_depth=20`), which improved RF from $188,582 to $187,248 ($1,334). Notably, feature engineering and feature selection actually *slightly hurt* RF (+$499 and +$155 respectively) — tree models already handle non-linearity and irrelevant features naturally. For **Ridge**, the pipeline provided far more benefit: from $244,905 baseline to $235,509 tuned (-$9,396 total), with tuning alpha=5000 contributing $3,217 and feature engineering (log_sqft) contributing $6,179. For **HGB**, gains were also modest ($191,490 to $190,954, only $537 total). The key takeaway: model selection (choosing RF over Ridge) contributed far more (~$48K) than any pipeline optimization step.

#### 5.B.2

Looking back at the hypotheses you proposed in **Milestone 1**, which were supported by your experimental results? Were any hypotheses disproved?

> **Supported:** (1) Log-transforming square footage improved Ridge by $6,144 (confirmed the non-linear sqft-price relationship). The engineered `log_sqft` feature also ranked 5th in Random Forest importance (0.0773), validating its predictive value. (2) Outlier capping at $5M improved ALL models substantially (Ridge -$21,753, HGB -$10,387, RF -$7,498), confirming that extreme luxury properties inflate MAE. **Partially supported:** (3) Bathroom density (`bath_density`) ranked 9th in feature importance (0.0342) but produced negligible MAE improvement in Part 2 (-$55 for HGB, +$491 for RF). It captures some signal but is largely redundant with existing size and bathroom features.

#### 5.B.3

Why did you select this model as your final model? Discuss both its predictive performance and any other considerations (such as stability, simplicity, or interpretability).

> We selected **Random Forest** (max_depth=20, n_estimators=200, min_samples_split=2) based on its consistently lowest CV MAE ($187,248) across all tuned models, outperforming HGB ($190,954) by $3,706 and Ridge ($235,509) by $48,261. The test set MAE of $191,757 is $4,509 higher than the CV estimate, slightly exceeding 1 standard deviation ($3,529). This modest gap is expected: (1) decision-making across Parts 2-4 was guided by CV scores, introducing slight optimistic bias, and (2) the test set is a single fixed 20% split that may contain a somewhat different property mix. Importantly, the gap is not large enough to suggest data leakage or fundamental overfitting — it represents normal generalization variance for a dataset of this size and heterogeneity.

#### 5.B.4

What did you learn about your dataset and the machine learning process through this end-to-end modeling workflow? If you had additional time, what would you investigate next?

> **Lessons learned:** (1) Non-linear models (RF, HGB) strongly outperform linear ones on heterogeneous real estate data — Ridge plateaus ~$48K worse regardless of tuning. (2) Feature engineering primarily benefits simpler models; tree-based models gain little from hand-crafted transformations. (3) For Random Forest, `max_depth` is the single most impactful hyperparameter, it converts a severely overfitting model (Part 1: $117K train-CV gap) into the best generalized performer. (4) The validation curve / plateau-start approach reliably identifies simpler models with near-optimal performance. (5) The $4,509 CV-to-test gap highlights the importance of not over-optimizing on validation scores.
>
> **Next steps:** (1) Log-transform the target variable (right-skewed prices likely inflate MAE for high-value properties). (2) Apply outlier capping in a production pipeline (showed $7-21K improvements but deferred for evaluation consistency). (3) Geographic clustering (lat/long to neighborhood features). (4) Target encoding for high-cardinality categoricals instead of ordinal encoding. (5) Model stacking (use Ridge/HGB predictions as meta-features for RF). (6) Expand hyperparameter search with Bayesian optimization (e.g., Optuna) to explore interactions more efficiently.